# Loan Approval Prediction - Exploratory Data Analysis

This notebook explores the `Loan_approval.csv` dataset:
- Dataset shape, data types, missing values
- Target distribution (`Loan_Status`)
- Correlation analysis
- Visualizations (Matplotlib & Seaborn)

> The actual cleaning / feature engineering used for modelling lives in `preprocessing.py` so that training and the Flask app stay in sync. This notebook is for exploration only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

df = pd.read_csv('../dataset/Loan_approval.csv')
df.head()

## 1. Dataset Shape

In [ ]:
print('Shape:', df.shape)
df.info()

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

## 3. Data Types

In [ ]:
df.dtypes

## 4. Target Distribution: `Loan_Status`

In [ ]:
print(df['Loan_Status'].value_counts())
print(df['Loan_Status'].value_counts(normalize=True).round(3))

plt.figure(figsize=(6, 4))
sns.countplot(x='Loan_Status', data=df, palette='viridis')
plt.title('Loan Status Distribution')
plt.show()

## 5. Numerical Feature Distributions

In [ ]:
numeric_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

## 6. Categorical Features vs Loan Status

In [ ]:
cat_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    sns.countplot(x=col, hue='Loan_Status', data=df, ax=ax, palette='Set2')
    ax.set_title(f'{col} vs Loan Status')
    ax.tick_params(axis='x', rotation=30)
fig.delaxes(axes.flatten()[-1])
plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
corr_cols = numeric_cols + ['Credit_History']
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap (Numerical Features)')
plt.show()

## 8. Data Quality Observations

A quick scan of unique values reveals inconsistent raw entries that need cleaning before modelling:
- `Gender`: contains typos like `Mle`, `Fmale`
- `Married`: mixes `Yes`/`No` with `Y`/`N`
- `Dependents`: contains `'four'` and `'3+'`
- `Self_Employed`: contains an invalid `'Self'` category
- `Property_Area`: contains `'semi-urban'` and `'Metro'` (non-standard)
- `Education`: contains `'Grad'` as a shorthand for `'Graduate'`
- `LoanAmount`: contains negative values and an outlier sentinel of `9999`
- `Loan_Amount_Term`: contains a sentinel value of `999`
- `Credit_History`: contains invalid codes like `-1` and `2` (valid values are only `0`/`1`)

All of this is handled centrally in `preprocessing.py -> clean_raw_values()`, which is used identically by both `train_model.py` and `app.py`.

In [ ]:
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Property_Area', 'Education']:
    print(col, '->', df[col].unique())

print('\nLoanAmount min/max:', df['LoanAmount'].min(), df['LoanAmount'].max())
print('Loan_Amount_Term unique:', sorted(df['Loan_Amount_Term'].dropna().unique()))
print('Credit_History unique:', df['Credit_History'].unique())